In [ ]:
# Student Name: Mohammad Abdalrouf Alnaghmoush
# Student ID: 2230004028


# Lab 13 — Part 2: Search Functions and Retrieval Approaches

**Learning goals:** Reuse the retrieval pipeline in a function, explore different queries, and compare sparse and dense retrieval.

## Setup — Run this cell first

The cell below recreates the small document collection and TF-IDF vectors used in Part 1.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]

vectorizer = TfidfVectorizer(stop_words="english")
document_vectors = vectorizer.fit_transform(documents)

## 1. Make a simple search function

Now combine the steps so you can search the document collection using any query.

In [ ]:
def search(query, k=2):
    """Rank documents by cosine similarity using the fitted TF-IDF vocabulary."""
    if not 1 <= k <= len(documents):
        raise ValueError("k must be between 1 and the number of documents.")
    query_vector = vectorizer.transform([query])
    similarity_scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = similarity_scores.argsort()[::-1][:k]
    results = [(int(i), float(similarity_scores[i]), documents[i]) for i in top_ids]
    print(f"Query: {query!r} | k={k}")
    for rank, (i, score, document) in enumerate(results, start=1):
        print(f"Rank {rank} | Document {i} | Score: {score:.3f} | {document}")
    return results

example_results = search("How can we measure similarity between text vectors?")


## 2. Sparse vs. dense retrieval (conceptual comparison)

| Sparse retrieval | Dense retrieval |
|---|---|
| Represents text using word-based features, e.g., TF-IDF | Represents text using learned embeddings |
| Works well when important query words appear in documents | Can identify similar meanings even when different words are used |
| Simple and lightweight | Usually requires a pretrained embedding model |

**Important limitation:** TF-IDF mainly matches words, not meanings. A query about *"automobiles"* may not retrieve a document about *"cars"* unless the words also overlap in other ways.

In modern RAG, retrieval can be sparse, dense, or a combination of both.

## 3. Does a high-ranked result always mean a good match?

The search function returns the **highest-scoring** documents, even when all the scores are zero. A high *rank* does not always mean a high *similarity score*.

For example, our collection is about NLP. What happens when we search for an unrelated topic like `"volcano eruption"`?

In [ ]:
search("volcano eruption", k=2)

# Compare it with a relevant query
print("--- Relevant query ---")
search("cosine similarity", k=2)

## 4. Improve search: skip results with a score of zero

We can improve the function by printing only documents with a **positive similarity score**.

In [ ]:
def search_filtered(query, k=2):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    top_ids = scores.argsort()[::-1][:k]

    found = False
    for i in top_ids:
        if scores[i] > 0:
            print(f"Score: {scores[i]:.3f} | {documents[i]}")
            found = True
    if not found:
        print("No matching documents found.")

search_filtered("volcano eruption")
print("---")
search_filtered("cosine similarity")

## 5. Practice questions

1. Use `search()` to find a note about **named entity recognition**. Which document ranks first?
2. Add `"BERT is a transformer model that uses bidirectional context."` to `documents`, rebuild the TF-IDF vectors using `vectorizer.fit_transform(documents)`, and search for `"transformer models"`. Why was rebuilding necessary?

In [ ]:
# Practice 1 — find the NER note.
ner_results = search("named entity recognition", k=2)
assert ner_results[0][0] == 3 and ner_results[0][1] > 0
print("\nAnswer: Document 3 ranks first: " + ner_results[0][2])


### Practice 2 — Add BERT, rebuild the index, and search again


In [ ]:
# Compare retrieval before and after updating the document collection.
print("Before adding the BERT document:")
before_results = search("transformer models", k=2)
old_shape = document_vectors.shape

new_document = "BERT is a transformer model that uses bidirectional context."
if new_document not in documents:
    documents.append(new_document)
# Refit the vectorizer on the expanded collection and rebuild every document vector.
document_vectors = vectorizer.fit_transform(documents)

print(f"\nTF-IDF shape before: {old_shape}")
print(f"TF-IDF shape after:  {document_vectors.shape}")
print("New vocabulary contains 'transformer':", "transformer" in vectorizer.vocabulary_)
print("\nAfter rebuilding the index:")
after_results = search("transformer models", k=2)
assert after_results[0][0] == documents.index(new_document)
assert after_results[0][1] > 0
assert document_vectors.shape[0] == len(documents)
print("\nAnswer: The new BERT document ranks first after rebuilding the index.")
print("Refitting learns new vocabulary terms and recalculates IDF weights across all documents.")


**Why rebuilding was necessary:** A fitted TF-IDF vectorizer stores the vocabulary and inverse document frequencies of its original collection. Appending a string alone does not add its vector to the retrieval matrix or update the vocabulary and IDF weights. Calling `fit_transform(documents)` rebuilds the complete index with consistent feature dimensions and updated statistics. In this example, `transformer` becomes a vocabulary term, whereas `models` may still be out of vocabulary because the new note contains singular `model`.

**Sparse vs. dense:** TF-IDF ranks lexical overlap; dense retrieval uses learned embeddings to compare meaning. The original lab asks for a conceptual comparison, so no embedding model or external corpus is required.
